### This is a minimal example for sanitizing sentences. 

In [1]:
"""Minimal example of sanitizing sentences."""

import os

from preempt import LegacyNERShim, Sanitizer
from preempt.distilbert_recognizer import DistilBertRecognizer

In [2]:
# Hugging Face model id or a local directory with an exported ONNX graph.
# See .env.example. This demo runs the full-precision graph, not the int8 one.
model_path = os.environ["PREEMPT_MODEL_PATH"]
recognizer = DistilBertRecognizer(model_path=model_path, max_length=512, backend="onnx", onnx_file_name="model.onnx")
# The examples below call ner_model.extract to show what was detected.
ner_model = LegacyNERShim(recognizer)

In [3]:
# Anyone with the key and tweak can reverse FPE names and amounts. Store them
# like any other secret; decrypting later needs the same pair. See .env.example.
key = os.environ["PREEMPT_KEY"]
tweak = os.environ["PREEMPT_TWEAK"]
sanitizer = Sanitizer(ner_model, key=key, tweak=tweak)

### Example for sanitizing names of people:

In [4]:
sentences = ["Ben Parker and John Doe went to the bank.", "Who was late today? Adam."]
extracted = ner_model.extract(sentences, entity_type="Name")
print("NER extraction:")
print(extracted)
# Expected output: {'Name': [['Ben Parker', 'John Doe'], ['Adam']]}

NER extraction:
{'Name': [['Ben Parker'], ['Adam']]}


In [5]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity="Name", epsilon=1)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Grzegorz Byrne and John Doe went to the bank.', 'Who was late today? Rakesh.']


In [6]:
extracted = ner_model.extract(sanitized_sentences, entity_type="Name")
print("NER extraction:")
print(extracted)

NER extraction:
{'Name': [['Grzegorz Byrne'], ['Rakesh']]}


In [7]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity="Name")
print("Desanitized sentences:")
print(desanitized_sentences)
# Expected output: ['Ben Parker and John Doe went to the bank.',
# 'Who was late today? Adam.']

Desanitized sentences:
['Ben Parker and John Doe went to the bank.', 'Who was late today? Adam.']


### Example for sanitizing currency values with FPE:

In [8]:
sentences = ["Ben withdrew $10,000 from the bank. Adam got $5,550 in a loan.", "I won $25 in the lottery."]
extracted = ner_model.extract(sentences, entity_type="Money")
distilbert_extracted = recognizer.recognize(sentences[0])
print(f"DistilBERT extraction:{distilbert_extracted}")
print("NER extraction:")
print(extracted)
# Expected output: {'Money': [['$10,000', '$5,550'], ['$25']]}

DistilBERT extraction:[EntitySpan(start=0, end=3, text='Ben', label='NAME', confidence=0.9944203495979309, source='distilbert'), EntitySpan(start=14, end=20, text='10,000', label='MONEY', confidence=1.0, source='currency'), EntitySpan(start=36, end=40, text='Adam', label='NAME', confidence=0.9998372793197632, source='distilbert'), EntitySpan(start=46, end=51, text='5,550', label='MONEY', confidence=1.0, source='currency')]
NER extraction:
{'Money': [['10,000', '5,550'], ['25']]}


In [9]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity="Money", epsilon=0.1)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Ben withdrew $43693850642 from the bank. Adam got $7243860616 in a loan.', 'I won $58359676 in the lottery.']


In [10]:
extracted = ner_model.extract(sanitized_sentences, entity_type="Money")
print("NER extraction:")
print(extracted)

NER extraction:
{'Money': [['43693850642', '7243860616'], ['58359676']]}


In [11]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity="Money")
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Ben withdrew $10000 from the bank. Adam got $5550 in a loan.', 'I won $25 in the lottery.']


### Example for sanitizing currency values with m-LDP:

In [12]:
sentences = ["Ben withdrew $1000 from the bank. Adam got $555 in a loan.", "I won $25 in the lottery."]
extracted = ner_model.extract(sentences, entity_type="Money")
print("NER extraction:")
print(extracted)

NER extraction:
{'Money': [['1000', '555'], ['25']]}


In [13]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity="Money", epsilon=0.1, use_fpe=False, use_mdp=True)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Ben withdrew $1090 from the bank. Adam got $2080 in a loan.', 'I won $31 in the lottery.']


In [14]:
print(sanitizer.entity_mapping)

[{'1000': '1090', '555': '2080'}, {'25': '31'}]


In [15]:
extracted = ner_model.extract(sanitized_sentences, entity_type="Money")
print("NER extraction:")
print(extracted)

NER extraction:
{'Money': [['1090', '2080'], ['31']]}


In [16]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity="Money", use_fpe=False, use_mdp=True)
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Ben withdrew $1000 from the bank. Adam got $555 in a loan.', 'I won $25 in the lottery.']


### Example for sanitizing age values with m-LDP:

In [17]:
sentences = ["Ben turned 15 years old today.", "I am 25 years old."]
extracted = ner_model.extract(sentences, entity_type="Age")
print("NER extraction:")
print(extracted)

NER extraction:
{'Age': [['15'], ['25']]}


In [18]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity="Age", epsilon=0.3, use_mdp=True, use_fpe=False)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Ben turned 18 years old today.', 'I am 27 years old.']


In [19]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity="Age")
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:


['Ben turned 15 years old today.', 'I am 25 years old.']


In [20]:
sentences = [
    "Passage: Ben Parker, a 72 Male immigrant, needs your legal views. He earns $34,000 annually. Hamid Aziz is a native here.",
    "Jose Santiago is a 35 year old immigrant who works at Meta.",
]
extracted = ner_model.extract(sentences, entity_type="Name")
print("NER extraction:")
print(extracted)
extracted = ner_model.extract(sentences, entity_type="Age")
print("NER extraction:")
print(extracted)
extracted = ner_model.extract(sentences, entity_type="Money")
print("NER extraction:")
print(extracted)

NER extraction:
{'Name': [['Ben Parker', 'Hamid Aziz'], ['Jose Santiago']]}


NER extraction:
{'Age': [['72'], ['35']]}


NER extraction:
{'Money': [['34,000'], []]}


In [21]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity="Name", use_fpe=True)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Passage: Grzegorz Byrne, a 72 Male immigrant, needs your legal views. He earns $34,000 annually. Szymon Ribeiro is a native here.', 'Saad Mata is a 35 year old immigrant who works at Meta.']


In [22]:
extracted = ner_model.extract(sanitized_sentences, entity_type="Name")
print("NER extraction:")
print(extracted)

NER extraction:
{'Name': [['Grzegorz Byrne', 'Szymon Ribeiro'], ['Saad Mata']]}


In [23]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity="Name")
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Passage: Ben Parker, a 72 Male immigrant, needs your legal views. He earns $34,000 annually. Hamid Aziz is a native here.', 'Jose Santiago is a 35 year old immigrant who works at Meta.']
